# Bankruptcy - prepare the demo tables

**Input** - the UCI [Taiwanese Bankruptcy Prediction](https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction) archive, downloaded once to `data/bankruptcy/raw/uci572/` and read from there afterwards.

**Output** - at the paths `configs/bankruptcy.yaml` declares:

| File | |
| --- | --- |
| `data/bankruptcy/train.csv` · `valid.csv` · `test.csv` | the fixed 60 / 20 / 20 split the pipeline reads |
| `data/bankruptcy/few_shot.csv` | the example rows a discovery run shows: 10 batches of 32, same columns as `train.csv` plus `batch` |
| `data/bankruptcy/screen_train.csv` · `screen_valid.csv` | the rows discovery's screen fits and scores on, same format as `train.csv` |
| `data/bankruptcy/column_mapping.csv` | sanitized name → the archive's name |
| `data/bankruptcy/column_descriptions.json` | what each column means - the archive's own header - for discovery |

**The data** - 6,819 Taiwanese companies (1999-2009), 95 financial ratios, 3.2% bankrupt.

**The scenario** - a team already models with the profitability / leverage / growth ratios and proposes adding the **cash-flow family**. Does it earn its place? The config lists the family under `features.new`; every other numeric column is the incumbent set. Two controls are planted among the candidates, so the screens have something known to catch:

| Column | What it is | Should fail |
| --- | --- | --- |
| `cand_dup_roa_c` | a near-copy of an incumbent ratio | the redundancy screen |
| `cand_noise` | pure noise | the signal screen |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The repo root, wherever this notebook is run from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from preprocessing import (
    build_sample,
    build_shot_batches,
    fetch_archive,
    resolve_path,
    sanitize_columns,
    stratified_split,
    write_splits,
)
from validation import load_config
from validation.data import resolve_features

pd.set_option("display.width", 160, "display.max_columns", 12)

In [2]:
# Input: the UCI archive, cached here after the first download.
URL = "https://archive.ics.uci.edu/static/public/572/taiwanese+bankruptcy+prediction.zip"
RAW_DIR = ROOT / "data" / "bankruptcy" / "raw" / "uci572"

# Output: written to the paths this config declares, and checked against it.
CONFIG = ROOT / "configs" / "bankruptcy.yaml"

SEED = 42                          # the split and the clustering
VALID_SIZE, TEST_SIZE = 0.2, 0.2   # 60 / 20 / 20
SHOTS, SHOT_BATCHES = 32, 10       # example rows per batch; one batch per discovery round
SCREEN_SIZE, SCREEN_BALANCE = {"train": 1500, "valid": 600}, True    # the screen rows; balanced

cfg = load_config(CONFIG)
target, id_col = cfg.data.target, cfg.data.id_cols[0]

## 1. Load the archive

Headers are written for humans (`Bankrupt?`, `ROA(C) before interest and depreciation before interest`), so they are sanitized to snake_case. The originals are kept in `mapping` and become each column's description.

In [3]:
raw = pd.read_csv(fetch_archive(URL, RAW_DIR, "data.csv"))
frame, mapping = sanitize_columns(raw)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
mapping.head()

6,819 rows x 96 columns


,original,column
0,Bankrupt?,bankrupt
1,ROA(C) before interest and depreciation befor...,roa_c_before_interest_and_depreciation_before_...
2,ROA(A) before interest and % after tax,roa_a_before_interest_and_pct_after_tax
3,ROA(B) before interest and depreciation after...,roa_b_before_interest_and_depreciation_after_tax
4,Operating Gross Margin,operating_gross_margin


## 2. Shape the table

Rename the target, check the cash-flow family is there, plant the two controls, and add a row id - the id is what the leakage check and the few-shot file use.

In [4]:
frame = frame.rename(columns={"bankrupt": target})

CASH_FLOW = [
    "cash_flow_rate", "cash_flow_per_share", "cash_reinvestment_pct", "cash_total_assets",
    "cash_current_liability", "cash_turnover_rate", "cash_flow_to_sales",
    "cash_flow_to_total_assets", "cash_flow_to_liability", "cfo_to_assets",
    "cash_flow_to_equity",
]
missing = [c for c in CASH_FLOW if c not in frame.columns]
assert not missing, f"expected cash-flow columns not found after sanitizing: {missing}"

# The planted controls are constructed, not read.
rng = np.random.default_rng(0)
anchor = "roa_c_before_interest_and_depreciation_before_interest"
frame["cand_dup_roa_c"] = frame[anchor] * (1 + rng.normal(scale=0.01, size=len(frame)))
frame["cand_noise"] = rng.normal(size=len(frame))
PLANTED = {
    "cand_dup_roa_c": "planted control: a near-copy of an incumbent ratio",
    "cand_noise": "planted control: pure noise, unrelated to the target",
}

frame.insert(0, id_col, np.arange(len(frame)))
frame.shape

(6819, 99)

## 3. Profile

Nothing is missing here, so the interesting columns are the near-constant ones: `net_income_flag` never varies, and the data quality gate will drop it.

In [5]:
features = [c for c in frame.columns if c not in {target, id_col}]
profile = pd.DataFrame({
    "dtype": frame[features].dtypes.astype(str),
    "n_unique": frame[features].nunique(),
    "missing_rate": frame[features].isna().mean().round(4),
})
print(f"{len(frame):,} rows, {len(features)} features")
print(f"target {target!r}: {frame[target].mean():.2%} positive "
      f"({int(frame[target].sum()):,} of {len(frame):,})")
print(f"missing: {frame[features].isna().to_numpy().mean():.1%} of feature cells, "
      f"{int((profile['missing_rate'] > 0).sum())} of {len(features)} columns affected")
profile[profile["n_unique"] <= 2]

6,819 rows, 97 features
target 'bankrupt': 3.23% positive (220 of 6,819)
missing: 0.0% of feature cells, 0 of 97 columns affected


,dtype,n_unique,missing_rate
liability_assets_flag,int64,2,0.0
net_income_flag,int64,1,0.0


## 4. Split 60 / 20 / 20

Stratified on the target, so each part keeps the base rate, with a fixed seed. This is the **only** split: the pipeline reads these three tables exactly as written and never re-splits. Discovery draws on train and valid only - the few-shot rows the proposer sees come from train, and its screen fits on train and scores on valid - so test is touched by nothing before the final verdict.

In [6]:
frames = stratified_split(frame, target, valid_size=VALID_SIZE, test_size=TEST_SIZE, seed=SEED)

pd.DataFrame({
    name: {"rows": len(part), "positives": int(part[target].sum()),
           "positive_rate": round(part[target].mean(), 4)}
    for name, part in frames.items()
}).T

,rows,positives,positive_rate
train,4091.0,132.0,0.0323
valid,1364.0,44.0,0.0323
test,1364.0,44.0,0.0323


## 5. Few-shot example rows

The rows a discovery run prints under every column of its prompt - the only concrete data the proposer ever sees. They are chosen per class by KMeans on train's incumbent columns, one row per cluster (16 per class), so they cover the table rather than its densest region, and both classes appear even at a 3% base rate. Batch *r* is shown in round *r*; batch *b* takes the *b*-th closest row of each cluster, so successive rounds see different rows from the same regions.

Saved as the rows themselves - the same columns as `train.csv`, plus `batch` - and read back and cleaned exactly as the splits are, so the prompt shows what the models see.

In [7]:
base, new = resolve_features(frames["train"], cfg.features, cfg.data)
continuous = cfg.discovery.continuous_columns
categorical = ([c for c in base if c not in set(continuous)] if continuous is not None
               else [c for c in cfg.discovery.categorical_columns if c in base])

batches = build_shot_batches(frames["train"], target, columns=base, categorical=categorical,
                             shots=SHOTS, batches=SHOT_BATCHES, seed=SEED)
few_shot = pd.concat([b.assign(batch=i) for i, b in enumerate(batches)], ignore_index=True)
few_shot = few_shot[["batch", *frames["train"].columns]]   # train.csv's columns, plus batch
assert few_shot[id_col].isin(frames["train"][id_col]).all()

print(f"{len(base)} incumbent columns ({len(categorical)} shown as coded categories), "
      f"{len(new)} candidates")
print(f"{len(batches)} batches x {len(batches[0])} rows, from {len(frames['train']):,} train rows")
few_shot.groupby("batch")[target].agg(rows="size", positives="sum").T

84 incumbent columns (0 shown as coded categories), 13 candidates
10 batches x 32 rows, from 4,091 train rows


batch,0,1,2,3,4,5,6,7,8,9
rows,32,32,32,32,32,32,32,32,32,32
positives,16,16,16,16,16,16,16,16,16,16


## 6. Screen sample

The rows discovery's screen fits each proposal on - a sample of **train** - and scores it on - a sample of **valid**. It is the cheap signal the proposer gets back every round; the decision is made later, on the full splits. When `SCREEN_BALANCE` is true each class contributes up to half the rows, so a rare class is kept whole instead of the handful a uniform draw would give. Saved as the rows themselves, in the same format as `train.csv`.

Scoring on valid means the proposer's feedback comes from valid rows, so valid stops being an independent check on what it proposes; test still is. To keep valid independent, draw both samples from disjoint rows of train. `discovery.screen_data: splits` in the config skips these files and uses the whole train and valid splits.

In [8]:
screen = {
    part: build_sample(frames[part], target, SCREEN_SIZE[part],
                       balance=SCREEN_BALANCE, seed=SEED)
    for part in ("train", "valid")
}

pd.DataFrame({
    part: {"rows": len(rows), "positives": int(rows[target].sum()),
           "positive_rate": round(rows[target].mean(), 4)}
    for part, rows in screen.items()
}).T

,rows,positives,positive_rate
train,882.0,132.0,0.1497
valid,344.0,44.0,0.1279


## 7. Check against the config, then write

`write_splits` refuses to write anything if the tables disagree with `configs/bankruptcy.yaml`: a candidate the config names but the table lacks, a missing or non-binary target, splits with different columns, or an id in two splits.

In [9]:
written = write_splits(cfg, frames, root=ROOT, mapping=mapping, extra_descriptions=PLANTED)
written["few_shot"] = resolve_path(cfg.discovery.few_shot_path, ROOT)
few_shot.to_csv(written["few_shot"], index=False)
for part, rows in screen.items():
    written[f"screen_{part}"] = resolve_path(cfg.discovery.screen_paths[part], ROOT)
    rows.to_csv(written[f"screen_{part}"], index=False)

for name, path in written.items():
    print(f"{name:<13} {path.relative_to(ROOT)}")

train         data/bankruptcy/train.csv
valid         data/bankruptcy/valid.csv
test          data/bankruptcy/test.csv
descriptions  data/bankruptcy/column_descriptions.json
mapping       data/bankruptcy/column_mapping.csv
few_shot      data/bankruptcy/few_shot.csv
screen_train  data/bankruptcy/screen_train.csv
screen_valid  data/bankruptcy/screen_valid.csv


Next, from the repo root:

```bash
autofe -c configs/bankruptcy.yaml                                  # the declared candidates
autofe -c configs/bankruptcy.yaml --set discovery.enabled=true     # plus LLM-proposed ones
```